# A02 · 计费、分母与三方账本

先修：公用31–34；A01。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

定义CTR=点击/展示，CVR=转化/点击，CTCVR=转化/展示，且转化必须属于点击路径时CTCVR=CTR×CVR。CPM=1000×花费/展示，CPC=花费/点击，CPA=花费/转化；eCPM是期望每千次展示货币，CPC报价的eCPM=1000×pCTR×CPC。ROAS=归因收入/广告花费，并非利润率。此例无平台手续费，媒体收款=广告主支付。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
a=ac.billing(); b=ac.billing(clicks=400)
display(pd.DataFrame([a,b],index=['200 clicks','400 clicks']))
assert abs(a['ctr']*a['cvr']-a['ctcvr'])<1e-12
assert abs(1000*a['ctr']*a['cpc']-a['cpm'])<1e-12
assert a['publisher_receipt']==80

,ctr,cvr,ctcvr,cpm,cpc,cpa,roas,advertiser_margin,publisher_receipt
200 clicks,0.02,0.04,0.0008,8.0,0.4,10.0,3.0,160.0,80.0
400 clicks,0.04,0.02,0.0008,8.0,0.2,10.0,3.0,160.0,80.0


## 解释与局限

只改变点击数，花费和转化不变：CTR上升、CPC下降、CVR下降，但CPA与ROAS不变。不能以CTR提升推出广告主利润提升；收入也不等于广告造成的增量收入。

## 查看源码，再做练习

加入10%平台手续费，写出广告主支出=平台收入+媒体收入的守恒式。试给转化分母改成展示，指出指标名称应如何变化。

In [3]:
import inspect
print(inspect.getsource(ac.billing))

def billing(impressions=10000, clicks=200, conversions=8, spend=80., revenue=240.):
    if not 0 < conversions <= clicks <= impressions or spend <= 0:
        raise ValueError('Require positive, nested event counts and spend')
    ctr, cvr = clicks/impressions, conversions/clicks
    return dict(ctr=ctr, cvr=cvr, ctcvr=conversions/impressions,
                cpm=1000*spend/impressions, cpc=spend/clicks,
                cpa=spend/conversions, roas=revenue/spend,
                advertiser_margin=revenue-spend, publisher_receipt=spend)



## 来源与下一步

A-CCR 33–41页；Lab补充计费与账本定义。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。